# Trees & Traversal — Hierarchical Structure

A **tree** is a connected acyclic graph: every node except the **root** has exactly one parent, and nodes with no children are **leaves**. A **binary tree** restricts each node to at most two children (left, right). Trees model hierarchy and bound search to the tree's **height** $h$, which for a balanced binary tree of $n$ nodes is $\Theta(\log n)$. Each traversal below records the visit order step by step so the front of exploration can be watched moving through the tree.

$$ n \le 2^{h+1}-1, \qquad h \ge \lceil \log_2(n+1) \rceil - 1. $$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from collections import deque
from IPython.display import display

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def make_player(n_steps, render, label='step'):
    slider = widgets.IntSlider(value=0, min=0, max=n_steps-1, description=label,
                               continuous_update=False, layout=widgets.Layout(width='60%'))
    play = widgets.Play(value=0, min=0, max=n_steps-1, interval=600)
    widgets.jslink((play, 'value'), (slider, 'value'))
    out = widgets.interactive_output(render, {'k': slider})
    display(widgets.HBox([play, slider]), out)

# Binary tree as a dict: node -> (left, right). None means absent.
TREE = {
    'A': ('B', 'C'),
    'B': ('D', 'E'),
    'C': ('F', 'G'),
    'D': (None, None),
    'E': ('H', None),
    'F': (None, None),
    'G': (None, 'I'),
    'H': (None, None),
    'I': (None, None),
}
ROOT = 'A'

# Assign (x, y) by in-order x-position and depth-based y.
def layout(tree, root):
    pos = {}; counter = [0]
    def place(node, depth):
        if node is None: return
        l, r = tree[node]
        place(l, depth+1)
        pos[node] = (counter[0], -depth); counter[0] += 1
        place(r, depth+1)
    place(root, 0)
    return pos

POS = layout(TREE, ROOT)

def draw_tree(visited, frontier, current, title, order_label=''):
    fig, ax = plt.subplots(figsize=(8, 5))
    for node, (l, r) in TREE.items():
        x, y = POS[node]
        for child in (l, r):
            if child is not None:
                cx, cy = POS[child]
                ax.plot([x, cx], [y, cy], color='lightgray', lw=1.5, zorder=1)
    for node in TREE:
        x, y = POS[node]
        if node == current:   c = 'tomato'
        elif node in visited: c = 'seagreen'
        elif node in frontier: c = 'gold'
        else:                 c = 'lightsteelblue'
        ax.add_patch(plt.Circle((x, y), 0.32, facecolor=c, edgecolor='k', zorder=2))
        ax.text(x, y, node, ha='center', va='center', fontsize=12, zorder=3)
    ax.text(0.02, 0.02, order_label, transform=ax.transAxes, fontsize=11, color='seagreen')
    xs = [p[0] for p in POS.values()]; ys = [p[1] for p in POS.values()]
    ax.set_xlim(min(xs)-0.7, max(xs)+0.7); ax.set_ylim(min(ys)-0.7, max(ys)+0.7)
    ax.set_title(title); ax.axis('off'); plt.show()

## Anatomy: Root, Depth, Height

A node's **depth** is its distance from the root; the tree's **height** is the maximum depth. These two quantities bound every traversal and search cost. The widget colors all nodes at a chosen depth, making the level structure explicit.

$$ \text{depth(root)} = 0, \qquad \text{height} = \max_{\text{node}} \text{depth(node)}. $$

In [ ]:
def depth_of(node, root=ROOT, d=0):
    if node == root: return 0
    # BFS to find depth
    q = deque([(root, 0)])
    while q:
        cur, dd = q.popleft()
        if cur == node: return dd
        l, r = TREE[cur]
        for ch in (l, r):
            if ch is not None: q.append((ch, dd+1))
    return -1

DEPTHS = {n: depth_of(n) for n in TREE}
HEIGHT = max(DEPTHS.values())

def show_depth(d):
    fig, ax = plt.subplots(figsize=(8, 5))
    for node, (l, r) in TREE.items():
        x, y = POS[node]
        for child in (l, r):
            if child is not None:
                cx, cy = POS[child]; ax.plot([x, cx], [y, cy], color='lightgray', lw=1.5, zorder=1)
    for node in TREE:
        x, y = POS[node]
        c = 'tomato' if DEPTHS[node] == d else 'lightsteelblue'
        ax.add_patch(plt.Circle((x, y), 0.32, facecolor=c, edgecolor='k', zorder=2))
        ax.text(x, y, node, ha='center', va='center', fontsize=12, zorder=3)
    xs=[p[0] for p in POS.values()]; ys=[p[1] for p in POS.values()]
    ax.set_xlim(min(xs)-0.7, max(xs)+0.7); ax.set_ylim(min(ys)-0.7, max(ys)+0.7)
    ax.set_title(f'depth {d} highlighted   (tree height = {HEIGHT})'); ax.axis('off'); plt.show()

d_s = widgets.IntSlider(value=1, min=0, max=HEIGHT, description='depth')
display(d_s, widgets.interactive_output(show_depth, {'d': d_s}))

IntSlider(value=1, description='depth', max=3)

Output()

## Breadth-First Traversal (Level Order)

BFS uses a **queue**: it visits the root, enqueues its children, then repeatedly dequeues a node and enqueues *its* children — sweeping the tree level by level. The frontier (gold) is exactly the queue contents. Step through to watch the wave move down one full level before the next.

$$ \text{visit order} = \text{nodes sorted by depth, left to right.} $$

In [3]:
def bfs_frames(tree, root):
    q = deque([root]); visited = []; frames = []
    frames.append((list(visited), list(q), None, 'enqueue root'))
    while q:
        node = q.popleft()
        visited.append(node)
        frames.append((list(visited), list(q), node, f'visit {node}'))
        for ch in tree[node]:
            if ch is not None: q.append(ch)
        if q:
            frames.append((list(visited), list(q), node, f'enqueue children of {node}: queue={list(q)}'))
    frames.append((list(visited), [], None, f'done: {"".join(visited)}'))
    return frames

frames_bfs = bfs_frames(TREE, ROOT)
def draw_bfs(k):
    visited, frontier, current, note = frames_bfs[k]
    draw_tree(set(visited)-({current} if current else set()), set(frontier), current,
              f'BFS step {k}/{len(frames_bfs)-1}: {note}',
              order_label='visited: ' + ' '.join(visited))
make_player(len(frames_bfs), draw_bfs)

Output()

## Depth-First Traversal — Pre / In / Post Order

DFS uses recursion (an implicit **stack**), plunging down one branch before backtracking. *When* a node is recorded determines the order: **pre-order** records on arrival, **in-order** between its two subtrees, **post-order** after both subtrees. Switch the order and watch the same descent produce three different sequences.

$$ \text{pre: } N\,L\,R \qquad \text{in: } L\,N\,R \qquad \text{post: } L\,R\,N. $$

In [4]:
def dfs_frames(tree, root, mode):
    visited = []; frames = []
    def rec(node):
        if node is None: return
        l, r = tree[node]
        if mode == 'pre':
            visited.append(node); frames.append((list(visited), node, f'record {node} (arrive)'))
        rec(l)
        if mode == 'in':
            visited.append(node); frames.append((list(visited), node, f'record {node} (between subtrees)'))
        rec(r)
        if mode == 'post':
            visited.append(node); frames.append((list(visited), node, f'record {node} (after subtrees)'))
    frames.append((list(visited), None, 'start at root'))
    rec(root)
    frames.append((list(visited), None, f'done: {"".join(visited)}'))
    return frames

mode_s = widgets.Dropdown(options=[('pre-order','pre'),('in-order','in'),('post-order','post')],
                          value='pre', description='order')
dfs_area = widgets.Output()
def relaunch_dfs(*_):
    frames = dfs_frames(TREE, ROOT, mode_s.value)
    def draw(k):
        visited, current, note = frames[k]
        draw_tree(set(visited)-({current} if current else set()), set(), current,
                  f'DFS-{mode_s.value} step {k}/{len(frames)-1}: {note}',
                  order_label='recorded: ' + ' '.join(visited))
    dfs_area.clear_output(wait=True)
    with dfs_area: make_player(len(frames), draw)
mode_s.observe(relaunch_dfs, 'value')
display(mode_s, dfs_area)
relaunch_dfs()

Dropdown(description='order', options=(('pre-order', 'pre'), ('in-order', 'in'), ('post-order', 'post')), valu…

Output()

## DFS Made Explicit — The Stack Behind the Recursion

The recursion's call stack can be replaced by an explicit **stack** of nodes: pop a node, record it, then push its children (right before left, so left is processed first). Visualizing the stack alongside the tree reveals that DFS *is* a stack-driven traversal, the mirror image of BFS's queue.

$$ \text{push right, then left} \Rightarrow \text{left popped first} \Rightarrow \text{pre-order.} $$

In [5]:
def dfs_stack_frames(tree, root):
    stack = [root]; visited = []; frames = []
    frames.append((list(visited), list(stack), None, 'push root'))
    while stack:
        node = stack.pop()
        visited.append(node)
        frames.append((list(visited), list(stack), node, f'pop & visit {node}'))
        l, r = tree[node]
        for ch in (r, l):                 # push right first so left is on top
            if ch is not None: stack.append(ch)
        if stack:
            frames.append((list(visited), list(stack), node, f'push children -> stack={list(stack)}'))
    frames.append((list(visited), [], None, f'done: {"".join(visited)}'))
    return frames

frames_ds = dfs_stack_frames(TREE, ROOT)
def draw_dfs_stack(k):
    visited, stack, current, note = frames_ds[k]
    fig, (axt, axs) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios':[3,1]})
    # tree
    for node, (l, r) in TREE.items():
        x, y = POS[node]
        for child in (l, r):
            if child is not None:
                cx, cy = POS[child]; axt.plot([x, cx],[y, cy], color='lightgray', lw=1.5, zorder=1)
    vis_set = set(visited) - ({current} if current else set())
    front = set(stack)
    for node in TREE:
        x, y = POS[node]
        if node == current: c='tomato'
        elif node in vis_set: c='seagreen'
        elif node in front: c='gold'
        else: c='lightsteelblue'
        axt.add_patch(plt.Circle((x, y), 0.32, facecolor=c, edgecolor='k', zorder=2))
        axt.text(x, y, node, ha='center', va='center', fontsize=12, zorder=3)
    xs=[p[0] for p in POS.values()]; ys=[p[1] for p in POS.values()]
    axt.set_xlim(min(xs)-0.7, max(xs)+0.7); axt.set_ylim(min(ys)-0.7, max(ys)+0.7)
    axt.set_title(f'step {k}/{len(frames_ds)-1}: {note}'); axt.axis('off')
    # stack (top = last element, drawn at top)
    for i, v in enumerate(stack):
        top = (i == len(stack)-1)
        axs.add_patch(plt.Rectangle((0, i), 1, 0.9, facecolor='gold' if top else 'lightsteelblue', edgecolor='k'))
        axs.text(0.5, i+0.45, v, ha='center', va='center', fontsize=12)
    if stack: axs.annotate('top', xy=(1.1, len(stack)-1+0.45), va='center', color='tomato', fontsize=9)
    axs.set_xlim(-0.3, 1.9); axs.set_ylim(-0.2, 7); axs.set_title('stack'); axs.axis('off')
    plt.tight_layout(); plt.show()
make_player(len(frames_ds), draw_dfs_stack)

Output()